<a href="https://colab.research.google.com/github/w4bo/AA2627-unibo-dcai/blob/main/slides/03-explainability.ipynb" target="_parent"><img src="img/colab-badge.svg" alt="Open In Colab"/></a>

---
title: "Model Explainability with SHAP"
---

In [ ]:
#| echo: false
%pip install -q shap

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


# Learning outcomes

By the end of this notebook, you should be able to:

- Explain the SHAP baseline and feature contributions.
- Describe how SHAP treats known and unknown features.
- Calculate a Shapley value in a two-feature example.
- Distinguish SHAP explanation from wrapper feature selection.
- Interpret global and local SHAP results.

# 1. Global and local explanations

Model explainability answers two related questions:

- **Global explanation:** which features does the fitted model rely on across many predictions?
- **Local explanation:** which feature values moved one prediction away from the model's baseline?

SHAP connects these views by calculating a contribution for every feature and every observation.

# 2. A fitted flight-delay model

The example uses a decision tree trained on five synthetic flight features. The model remains fixed throughout the SHAP analysis.

The synthetic data keeps the example reproducible. The SHAP procedure is the same for a tree trained on real flight data.

In [ ]:
#| echo: false
rng = np.random.default_rng(RANDOM_STATE)
n_flights = 2_000

bad_weather = rng.binomial(1, 0.25, n_flights)
departure_hour = rng.integers(0, 24, n_flights)
is_holiday = rng.binomial(1, 0.12, n_flights)
scheduled_duration_min = np.clip(rng.normal(120, 35, n_flights), 40, 300)
late_departure_probability = np.clip(
    0.10 + 0.45 * bad_weather + 0.15 * is_holiday,
    0,
    0.95,
)
late_departure = rng.binomial(1, late_departure_probability)

logit = (
    -2.3
    + 2.2 * bad_weather
    + 1.8 * late_departure
    + 0.7 * is_holiday
    + 0.006 * (scheduled_duration_min - 120)
    + 0.03 * np.abs(departure_hour - 12)
)
delay_probability = 1 / (1 + np.exp(-logit))
delayed = rng.binomial(1, delay_probability)

X = pd.DataFrame({
    "bad_weather": bad_weather,
    "late_departure": late_departure,
    "departure_hour": departure_hour,
    "is_holiday": is_holiday,
    "scheduled_duration_min": scheduled_duration_min,
})
y = pd.Series(delayed, name="delayed")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=30,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

pd.DataFrame({
    "metric": ["balanced accuracy", "F1"],
    "value": [
        balanced_accuracy_score(y_test, y_pred),
        f1_score(y_test, y_pred),
    ],
})


# 3. SHAP values

SHAP explains how a **fixed, already-trained model** produced a prediction.

For one observation, SHAP divides the difference between the model's baseline output and its prediction among the features:

$$f(x) = E[f(X)] + \phi_1 + \phi_2 + \cdots + \phi_p$$

- $E[f(X)]$ is the **baseline**, the model's expected output for the reference population.
- $\phi_j$ is the contribution assigned to feature $j$ for this observation.
- Positive contributions increase the output. Negative contributions decrease it.

SHAP explains the model's behavior. A large contribution does not prove that the feature caused the outcome.

# 4. The SHAP baseline

Choose a representative **reference dataset**, ask the model to predict every row, and average those outputs:

$$E[f(X)] \approx \frac{1}{n}\sum_{i=1}^{n}f(x_i)$$

| Reference flight | Predicted delay probability |
|---|---:|
| A | 0.10 |
| B | 0.20 |
| C | 0.30 |
| D | 0.60 |

The baseline is $(0.10 + 0.20 + 0.30 + 0.60)/4 = 0.30$. SHAP starts every explanation from this model output.

The baseline is the average **prediction**, not the observed delay rate. The two may differ when the model is not perfectly calibrated.

# 5. Known and unknown features

SHAP reveals the feature values of the observation being explained one group at a time.

- A **known feature** is fixed to this observation's value.
- An **unknown feature** is averaged out using the reference distribution. SHAP does not replace it with zero.

Consider a flight with bad weather and a late departure:

| Information revealed | Expected model output |
|---|---:|
| Nothing about this flight | $v(\varnothing)=0.30$ |
| Bad weather only | $v(W)=0.50$ |
| Late departure only | $v(L)=0.40$ |
| Both feature values | $v(W,L)=0.70$ |

$v(S)$ is the expected model output when the features in set $S$ are fixed and the remaining features are averaged out.

# 6. Computing a SHAP contribution

A feature's effect can depend on which other features are already known. SHAP therefore averages its marginal contribution across the possible feature orders.

For **weather**:

$$W \text{ first}: \quad v(W)-v(\varnothing)=0.50-0.30=0.20$$

$$W \text{ after }L: \quad v(W,L)-v(L)=0.70-0.40=0.30$$

$$\phi_W = \frac{0.20+0.30}{2}=0.25$$

Similarly, $\phi_L=0.15$. The contributions reconstruct the prediction:

$$0.70 = 0.30 + 0.25 + 0.15$$

# 7. All feature contexts

For feature $j$, exact Shapley values consider every subset $S$ that does not contain $j$:

$$\phi_j = \sum_{S \subseteq N \setminus \{j\}} \frac{|S|!(M-|S|-1)!}{M!} \left[v(S \cup \{j\})-v(S)\right]$$

The bracketed term measures what changes when feature $j$ becomes known in context $S$. The weight treats every possible feature order equally.

With $M$ features, each feature has $2^{M-1}$ possible subsets of the other features. For 10 features this is 512 subsets per feature. For 20 features it is 524,288. Direct enumeration quickly becomes impractical.

# 8. How the SHAP library performs the calculation

SHAP normally keeps the fitted model fixed. It does **not retrain one model for every feature subset**.

- `TreeExplainer` uses the fitted tree's paths and stored sample counts to calculate contributions efficiently.
- `LinearExplainer` uses the fitted coefficients and the background feature distribution.
- `KernelExplainer` and `PermutationExplainer` approximate the result by evaluating selected coalitions or feature orders with the existing prediction function.

This differs from wrapper selection such as SFS. SFS repeatedly retrains models to choose features. SHAP explains predictions from one fitted model.

The reference data still matters: changing the background population can change the baseline and the SHAP contributions.

# 9. Global SHAP importance

A local SHAP value explains one prediction. Global importance averages the absolute local contributions over many observations:

$$I_j = \frac{1}{n}\sum_{i=1}^{n}|\phi_{ij}|$$

Absolute values prevent positive and negative effects from cancelling. The result measures how strongly the fitted model uses each feature on average.

In [ ]:
#| echo: false
import shap

X_shap = X_test.reset_index(drop=True)
explainer = shap.TreeExplainer(tree)
raw_shap_values = explainer.shap_values(X_shap)

# Different SHAP versions return binary-class explanations in different shapes.
if isinstance(raw_shap_values, list):
    shap_values_positive_class = raw_shap_values[1]
else:
    shap_values_array = np.asarray(raw_shap_values)
    if shap_values_array.ndim == 3:
        shap_values_positive_class = shap_values_array[:, :, 1]
    else:
        shap_values_positive_class = shap_values_array

expected_values = np.asarray(explainer.expected_value)
baseline_positive_class = float(
    expected_values[1] if expected_values.size > 1 else expected_values.item()
)
print(f"SHAP baseline for the delayed class: {baseline_positive_class:.3f}")

shap_importance = (
    pd.DataFrame({
        "feature": X_shap.columns,
        "mean_abs_shap": np.abs(shap_values_positive_class).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
display(shap_importance)

shap.summary_plot(
    shap_values_positive_class,
    X_shap,
    plot_type="bar",
    show=False,
)
plt.title("Global SHAP importance for the delayed class")
plt.tight_layout()
plt.show()


# 10. Local SHAP explanation

Global importance summarizes many predictions.

- A local SHAP explanation focuses on one row and shows which selected features pushed the model toward or away from the delayed-flight class.
- Positive SHAP values increase the delayed-class output for this observation.
- Negative SHAP values decrease it.

In [ ]:
#| echo: false
row_id = 0
predicted_delay_probability = tree.predict_proba(X_shap.iloc[[row_id]])[0, 1]
reconstructed_output = (
    baseline_positive_class + shap_values_positive_class[row_id].sum()
)

display(pd.DataFrame({
    "quantity": [
        "baseline",
        "sum of SHAP contributions",
        "reconstructed output",
        "model prediction",
    ],
    "value": [
        baseline_positive_class,
        shap_values_positive_class[row_id].sum(),
        reconstructed_output,
        predicted_delay_probability,
    ],
}))

local_shap = (
    pd.DataFrame({
        "feature": X_shap.columns,
        "feature_value": X_shap.iloc[row_id].to_numpy(),
        "shap_contribution": shap_values_positive_class[row_id],
    })
    .assign(abs_contribution=lambda frame: frame["shap_contribution"].abs())
    .sort_values("abs_contribution", ascending=False)
    .drop(columns="abs_contribution")
    .reset_index(drop=True)
)
local_shap
